In [1]:

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Install Libraries

In [2]:
 !pip install diffusers transformers accelerate fastapi uvicorn pyngrok pillow -q
 
print("all installations successful")

all installations successful


* diffusers: library that runs stable diffusion
* transformers: Hugging Face's core library (used by both SD and BLIP)
* accelerate: increases speed of gpu inference
* fastapi: simple web server
* uvicorn: runs fastapi server
* pyngrok: python wrapper for ngrok
* python-multipart: lets fastapi get images

# imports


In [3]:
import torch
from diffusers import StableDiffusionPipeline, StableDiffusionImg2ImgPipeline
from transformers import BlipProcessor, BlipForConditionalGeneration, CLIPImageProcessor
from PIL import Image
import io, base64
import os
import threading 
import time
print("imports successful")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


imports successful


* diffusers is Hugging Face's library for image generation models Pipeline = a complete ready-to-use model with all steps pre-connected StableDiffusionPipeline = takes text → makes image StableDiffusionImg2ImgPipeline = takes image + text → makes new image
* Blip is a model that sees image and geneerates text, Blipprocessor prepares image before feeding it to model, BlipForConditionalGeneration is the model that  generate text for image
* io = handles file-like objects in memory (like a fake file), base64 = converts images to text format so we can send them over internet

# Loading required models

In [4]:
#txt2img pipeline
txt2img_pipe= StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
torch_dtype= torch.float16,
safety_checker= None
)

txt2img_pipe= txt2img_pipe.to("cuda")
print("Stable Diffusion model for text to image is successfully loaded..")


#img2img pipeline
img2img_pipe= StableDiffusionImg2ImgPipeline(
    vae= txt2img_pipe.vae,
    text_encoder= txt2img_pipe.text_encoder,
    tokenizer= txt2img_pipe.tokenizer,
    unet= txt2img_pipe.unet,
    scheduler= txt2img_pipe.scheduler,
    safety_checker= None,
    feature_extractor=CLIPImageProcessor.from_pretrained(
        "runwayml/stable-diffusion-v1-5", 
        subfolder="feature_extractor"
    )
).to("cuda")

print("img2img model is also ready")

blip_processor= BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)
blip_model= BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base",
    torch_dtype= torch.float16
).to("cuda")

print("all models ready")

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its resul

...

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


all models ready


# Create FastAPI Server

In [5]:
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import JSONResponse
import uvicorn
from pyngrok import ngrok

In [6]:
#converts PIL image to base64 string 
def img2base64(pil_image):   # This function takes a PIL image and returns a base64 string We use this because we can't send an image directly in JSON
    buffer = io.BytesIO()           # Create an empty "file" in memory (not on disk)
    pil_image.save(buffer, format="PNG")  # Save the image INTO that memory file
    buffer.seek(0)                  # Go back to the start of the memory file
    img_bytes = buffer.read()       # Read all the bytes
    img_b64 = base64.b64encode(img_bytes)     # Convert bytes to base64
    return img_b64.decode("utf-8")  # Convert bytes to a regular string


#style presets
STYLE_PRESETS = {
    "Realistic":  "",   # No addition = default realistic
    "Anime":      ", anime style, vibrant colors, Studio Ghibli",
    "Oil Painting": ", oil painting, thick brush strokes, impressionist",
    "Watercolor": ", watercolor painting, soft edges, pastel colors",
    "Cyberpunk":  ", cyberpunk, neon lights, dark city, futuristic",
    "Sketch":     ", pencil sketch, black and white, hand drawn",
}

#creating fastapi app
app= FastAPI()
# @app.post("/generate") means: when someone sends a POST request to /generate, run this function
@app.get("/health")
def health():
    return {
        "status": "ok",
        "gpu": torch.cuda.get_device_name(0),
        "model_loaded": txt2img_pipe is not None
    }

@app.post("/generate")
# POST is used when we're sending information — here we send the prompt, style etc.
async def generate_image(
    prompt: str = Form(...),        # The text the user typed
    style: str = Form("Realistic"), # Which style they chose (default: Realistic)
    steps: int = Form(20),          # How many denoising steps (more = better quality but slower)
    batch: int = Form(1)            # How many images to generate
):
    # Build the final prompt by adding style text
    style_text = STYLE_PRESETS.get(style, "")  
    # .get(key, default) = get value from dictionary, or use default if key not found
    full_prompt = prompt + style_text

    
    negative_prompt = "blurry, bad quality, ugly, deformed, low resolution"

    results=[]


    for i in range(batch):
        output= txt2img_pipe(
            prompt= full_prompt,
            negative_prompt= negative_prompt,
            num_inference_steps= steps,
            guidance_scale= 7.5,
            height= 512,
            width= 512
        )

        img= output.images[0]
        results.append(img2base64(img))

    # Return a JSON response — JSON is just a structured text format
    # Streamlit will receive this JSON and decode the images
    return JSONResponse({"images": results, "prompt_used": full_prompt})

# Image to Image endpoint
@app.post("/img2img")
async def img_to_img(
    file: UploadFile = File(...),   # The uploaded image file
    prompt: str = Form(...),        # Style description
    strength: float = Form(0.7)):
    try:
        # Read the uploaded image bytes and convert to PIL
        image_bytes = await file.read()  # await = wait for the file to fully upload
        init_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        # .convert("RGB") = make sure it's RGB (not RGBA which has transparency)
        
        # Resize to 512x512 (SD works best at this size)
        init_image = init_image.resize((512, 512))
        
        output = img2img_pipe(
            prompt=prompt,
            image=init_image,
            strength=strength,   # How much to change the original image
            num_inference_steps=30,
            guidance_scale=7.5,
        )
        
        result_img = output.images[0]
        return JSONResponse({"image": image2base64(result_img)})

    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)  # status_code=500 = HTTP "Internal Server Error"

# Image captioning endpoint
@app.post("/caption")
async def caption_image(file: UploadFile = File(...)):
    try:
        image_bytes = await file.read()
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        
        # BlipProcessor prepares the image (resizes, normalizes pixel values)
        # return_tensors="pt" means "return PyTorch tensors" (the format the model expects)
        inputs = blip_processor(image, return_tensors="pt").to("cuda", torch.float16)
        
        # Generate the caption
        # max_new_tokens=50 = generate at most 50 words
        output = blip_model.generate(**inputs, max_new_tokens=50)
        
        # Decode converts the model's number output back to text
        caption = blip_processor.decode(output[0], skip_special_tokens=True)
        # skip_special_tokens=True = remove internal tokens like [SEP], [CLS] etc.
        
        return JSONResponse({"caption": caption})

    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)

# Create Ngrok Tunnel

In [11]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
ngrok_token = user_secrets.get_secret("NGROK_TOKEN")

In [12]:

if not ngrok_token:
    print("WARNING: NGROK_TOKEN secret not set. Add it in Kaggle → Add-ons → Secrets")
else:
    ngrok.set_auth_token(ngrok_token)

tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print(f"\n{'='*55}")
print(f"  COPY THIS URL INTO YOUR STREAMLIT APP:")
print(f"  {public_url}")
print(f"{'='*55}\n")


def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")



server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
# Actually start the thread — run_server() begins executing in the background

time.sleep(3)

print("Server is live. Do NOT stop this cell.")
print("Press the Stop (square) button in Kaggle to shut down.\n")

try:
    while True:
        # while True = loop forever (until interrupted)
        time.sleep(60)
        print(f"[alive] VRAM: {torch.cuda.memory_allocated(0)/1024**3:.1f}GB | {public_url}")

except KeyboardInterrupt:
    # KeyboardInterrupt = when you press Stop in Kaggle
    print("Shutting down...")
    ngrok.kill()


  COPY THIS URL INTO YOUR STREAMLIT APP:
  https://imitate-floral-jaunt.ngrok-free.dev

Server is live. Do NOT stop this cell.
Press the Stop (square) button in Kaggle to shut down.

Shutting down...
